# Optimizing Rank-Order Doctor-to-Hospital Assignments: Greedy vs. Linear Sum Assignment

#### **Authors:** Arnav Mankad, Bobby Wang, Moria Li

## Problem

The goal of this project is to assign $N$ doctors to $K$ hospitals based on each doctor's ranked preferences while respecting hospital capacity constraints. Each doctor ranks the hospitals from most to least preferred, where a lower rank represents a more preferred hospital.

Let

* $r_{nk}$ = rank assigned by doctor $n$ to hospital $k$
* $c_k$ = maximum capacity of hospital $k$
* $x_{nk} = 1$ if doctor $n$ is assigned to hospital $k$, and $0$ otherwise

The objective is to minimize the total preference rank across all doctors:

$$
\min \sum_{n=1}^{N}\sum_{k=1}^{K} r_{nk}x_{nk}
$$

subject to each doctor being assigned to exactly one hospital:

$$
\sum_{k=1}^{K} x_{nk} = 1
\qquad \forall n
$$

and each hospital remaining within its capacity:

$$
\sum_{n=1}^{N} x_{nk} \leq c_k
\qquad \forall k
$$

with

$$
x_{nk} \in \{0,1\}.
$$

## Approaches

We will compare two approaches to solving the assignment problem.

**Greedy Baseline:** Doctors are processed sequentially. Each doctor is assigned to their highest-ranked hospital that still has available capacity. Once an assignment is made, it is not reconsidered. This provides a simple baseline but does not necessarily minimize the total preference rank across all doctors because each assignment is made based only on the current doctor's preference without considering its effect on later assignments.

**Linear Sum Assignment:** We originally intended to use the Hungarian method, a classical algorithm for solving the linear assignment problem. For the actual implementation, we will use SciPy's `scipy.optimize.linear_sum_assignment` function, which solves the problem using a modified Jonker–Volgenant algorithm. While both algorithms solve the same optimization problem, the Jonker–Volgenant algorithm uses a different and generally more efficient procedure for finding the optimal assignment. Doctor preference ranks are used as assignment costs, so minimizing the total cost corresponds to minimizing the total preference rank. Because hospitals may accept multiple doctors, each hospital is represented by multiple assignment slots according to its capacity. This converts the capacity-constrained problem into a linear sum assignment problem that can be solved using `linear_sum_assignment`. This optimization-based approach provides a globally optimal solution for the defined objective and can be directly compared with the simpler greedy baseline.

## Evaluation

The greedy and linear sum assignment approaches will be run on the same sets of doctor preferences and hospital capacities. Their resulting assignments will be compared using three metrics:

1. **Total Preference Rank**

$$
R_{\text{total}} = \sum_{n=1}^{N} r_{n,a(n)}
$$

where $a(n)$ is the hospital assigned to doctor $n$. A lower total rank indicates a better overall assignment.

2. **Average Assigned Rank**

$$
R_{\text{avg}} = \frac{R_{\text{total}}}{N}
$$

This measures the average preference rank received by a doctor.

3. **Percentage Receiving First Choice**

$$
P_{\text{first}} =
\frac{\text{Number of doctors assigned to their first choice}}{N}
\times 100
$$

A higher percentage indicates that more doctors received their most-preferred hospital.

We will generate test cases with different numbers of doctors, hospital capacities, and randomly generated preference rankings. Both approaches will receive the same input for each test so that their performance can be directly compared.

## Assumptions and Constraints

The assignment problem uses the following assumptions:

* Each doctor provides a complete ranking of all $K$ hospitals.
* Rank $1$ represents a doctor's most-preferred hospital.
* Each doctor must be assigned to exactly one hospital.
* Each hospital has a fixed maximum capacity $c_k$.
* Total hospital capacity is sufficient to assign every doctor:

$$
\sum_{k=1}^{K} c_k \geq N
$$

* Hospitals do not have preferences over doctors.
* Hospitals do not need to be filled to their maximum capacity.
* Rankings represent preference order but not the strength of preference between hospitals.

In [1]:
def greedy_assignment(preferences, capacities):
    """
    Assign doctors to hospitals using a greedy approach.

    Doctors are processed sequentially. Each doctor is assigned to their
    highest-ranked hospital that still has available capacity.

    Parameters
    ----------
    preferences : list[list[int]]
        Each doctor's hospital preferences, ordered from most to least preferred.

    capacities : list[int]
        Maximum number of doctors each hospital can accept.

    Returns
    -------
    assignments : list[int]
        Hospital assigned to each doctor.
    total_rank : int
        Sum of all assigned preference ranks.
    average_rank : float
        Average preference rank received.
    first_choice_percentage : float
        Percentage of doctors who receive their first choice.
    """

    n_doctors = len(preferences)

    # Track available capacity without modifying the original list
    remaining_capacity = capacities.copy()

    # -1 indicates that a doctor has not yet been assigned
    assignments = [-1] * n_doctors
    assigned_ranks = []

    # Process doctors sequentially
    for doctor in range(n_doctors):

        # Check hospitals from most to least preferred
        for rank, hospital in enumerate(preferences[doctor], start=1):

            if remaining_capacity[hospital] > 0:
                assignments[doctor] = hospital
                remaining_capacity[hospital] -= 1
                assigned_ranks.append(rank)
                break

        # This should not occur if total capacity is sufficient
        if assignments[doctor] == -1:
            raise ValueError(
                f"Doctor {doctor} could not be assigned. "
                "Check hospital capacities."
            )

    # Calculate performance metrics
    total_rank = sum(assigned_ranks)
    average_rank = total_rank / n_doctors

    first_choices = sum(rank == 1 for rank in assigned_ranks)
    first_choice_percentage = 100 * first_choices / n_doctors

    return (
        assignments,
        total_rank,
        average_rank,
        first_choice_percentage
    )

## Linear Sum Assignment


## Test Data & Comparison
ask chatgpt to generate test data for this problem (a method that takes n and k and returns random doctor preference lists and hospital preferences)
write a script to compare the two methods with the generated test data

## References
- García, A. Greedy algorithms: a review and open problems. J Inequal Appl 2025, 11 (2025). https://doi.org/10.1186/s13660-025-03254-1
- Kuhn, H.W. (1955), The Hungarian method for the assignment problem. Naval Research Logistics, 2: 83-97. https://doi.org/10.1002/nav.3800020109